# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [ ]:
import os
from glob import glob
from dotenv import load_dotenv
import dask.dataframe as dd
import pandas as pd
import random


In [ ]:
load_dotenv()
PRICE_DATA = os.getenv('PRICE_DATA')
print(PRICE_DATA)
print(os.listdir(PRICE_DATA))


../../05_src/data/prices/
['EDOG', 'BKTI', 'APTX', 'FNDF', 'CTBI', 'AOSL', 'INFY', 'UVE', 'CNSP', 'WRLSU', 'BLK', 'CTSO', 'IBTG', 'CLNC', 'AMAT', 'MICT', 'IQV', 'JHG', 'TERP', 'PSB', 'ZIONL', 'RPG', 'CTO', 'TERM', 'TNAV', 'FENC', 'MDRRP', 'ATHM', 'RILYP', 'CVCO', 'ZNH', 'GLG', 'CIK', 'PNR', 'LYV', 'MFO', 'A', 'BSE', 'SONA', 'ESGRP', 'DTW', 'MAV', 'PCB', 'HCKT', 'SO', 'CYTK', 'RBBN', 'EVRI', 'KNX', 'AMPY', 'BRC', 'AZO', 'IMBI', 'GPC', 'LIVE', 'TMSR', 'TRXC', 'FDUSZ', 'LAZY', 'RTW', 'COTY', 'HCM', 'REFR', 'KZR', 'DK', 'AXP', 'IPDN', 'IRBT', 'FTEC', 'PSN', 'IX', 'SIZE', 'POWL', 'UEC', 'KRC', 'AWRE', 'JAX', 'GRF', 'CVCY', 'TDJ', 'KEN', 'ALDX', 'EFNL', 'SGMO', 'ACB', 'ZEUS', 'SMBC', 'TFSL', 'HELE', 'CHRS']


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
# Write your code below.

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)



['../../05_src/data/prices/EDOG/EDOG_2020/part.0.parquet', '../../05_src/data/prices/EDOG/EDOG_2020/part.1.parquet', '../../05_src/data/prices/EDOG/EDOG_2018/part.0.parquet', '../../05_src/data/prices/EDOG/EDOG_2018/part.1.parquet', '../../05_src/data/prices/EDOG/EDOG_2016/part.0.parquet', '../../05_src/data/prices/EDOG/EDOG_2016/part.1.parquet', '../../05_src/data/prices/EDOG/EDOG_2017/part.0.parquet', '../../05_src/data/prices/EDOG/EDOG_2017/part.1.parquet', '../../05_src/data/prices/EDOG/EDOG_2019/part.0.parquet', '../../05_src/data/prices/EDOG/EDOG_2019/part.1.parquet', '../../05_src/data/prices/EDOG/EDOG_2015/part.0.parquet', '../../05_src/data/prices/EDOG/EDOG_2015/part.1.parquet', '../../05_src/data/prices/EDOG/EDOG_2014/part.0.parquet', '../../05_src/data/prices/EDOG/EDOG_2014/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet', '../../05_src

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
dd_feat = dd.read_parquet(parquet_files)
dd_feat = dd_feat.set_index('Date').map_partitions(lambda df: df.sort_index())
dd_feat['Close_lag_1'] = dd_feat.groupby('ticker')['Close'].shift(1)
dd_feat['Adj_Close_lag_1'] = dd_feat.groupby('ticker')['Adj Close'].shift(1)
dd_feat['returns'] = (dd_feat['Close'] / dd_feat['Close_lag_1']) - 1
dd_feat['hi_lo_range'] = dd_feat['High'] - dd_feat['Low']
dd_feat.head(15)





+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.

df_feat = dd_feat.compute()
df_feat = df_feat.sort_values(['ticker', 'Date'])
df_feat['returns_ma_10'] = df_feat.groupby('ticker')['returns'].rolling(10).mean().reset_index(level=0, drop=True)
df_feat.head(15)



Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

1. It wasn't necessary to convert to pandas as Dask consists of a function that could calculate the necessary value (i.e. - groupby().rolling().mean())
2. It would have been better to do it in Dask as it provides the advantage of memory efficiency, scalability, and parallel computation

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.